# ESS Round 4 data curation: CCA-ready outputs with and without official ESS weights

This notebook reproduces the deterministic data-curation stage of Jochem van Noord's ESS Round 4 workflow in Python and brings the result into the same respondent-level format used for the ESS Round 8 curation.

The notebook:

- reads `data/ESS4e04_6/ESS4e04_6.csv`;
- constructs the 19 Round 4 belief variables defined in the supplied `Data cleaning_ESS4.R` script;
- retains respondents aged 18 or older, while retaining respondents whose age is missing;
- retains respondents with no more than two missing constructed beliefs;
- copies the official ESS weights without recalculating them;
- creates separate outputs with and without the four weight columns;
- preserves the raw ESS row number in `ess_row_id` and uses `ess_unique_id` for respondent matching.

The later notebook `03_validation_and_cross_round_checks.ipynb` will compare this Python result directly with Van Noord's saved `df_ESS4.RData`. The present notebook validates all deterministic dimensions and internal consistency but does not yet claim exact row-by-row agreement with the R object.

## Step 1 — Locate the project, import the shared code, and define paths

Run this notebook from either the project root or the `notebooks/` folder. The first code cell locates the root folder containing `data/`, `notebooks/`, and `src/`, adds it to Python's import path, and imports the shared curation functions and the ESS4 configuration.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if all((candidate / folder).is_dir() for folder in ("data", "notebooks", "src")):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing data/, notebooks/, and src/."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ess4_config as config
from src.ess_curation_common import (
    apply_cca_sample_rule,
    construct_belief_variables,
    require_columns,
    split_weighted_and_unweighted,
    summarise_beliefs,
    valid_range,
    validate_weight_split,
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / config.RAW_DATA_FOLDER
    / config.RAW_DATA_FILENAME
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

WITHOUT_WEIGHTS_PATH = PROCESSED_DIR / config.OUTPUT_FILENAMES["without_weights"]
WITH_WEIGHTS_PATH = PROCESSED_DIR / config.OUTPUT_FILENAMES["with_weights"]

print("Project root:", PROJECT_ROOT)
print("Raw ESS4 file:", RAW_DATA_PATH)
print("Processed-data folder:", PROCESSED_DIR)

Project root: /Users/karan/Desktop/SSM-MERC/polarization/data_curation
Raw ESS4 file: /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/ESS4e04_6/ESS4e04_6.csv
Processed-data folder: /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed


## Step 2 — Load the raw ESS Round 4 file and verify the required inputs

No rows or values are changed here. The checks confirm the expected raw sample size, country count, required metadata, constituent belief items, and the four official ESS weight columns.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)

require_columns(
    raw,
    config.REQUIRED_RAW_COLUMNS,
    context="raw ESS Round 4 file",
)

assert len(raw) == config.EXPECTED_RAW_N, (
    f"Expected {config.EXPECTED_RAW_N:,} raw rows, found {len(raw):,}."
)
assert raw["cntry"].nunique(dropna=True) == config.EXPECTED_RAW_COUNTRIES

weight_missing_raw = raw.loc[:, config.WEIGHT_COLUMNS].isna().sum()
assert (weight_missing_raw == 0).all(), (
    "Unexpected missing values in raw ESS weights: "
    f"{weight_missing_raw.to_dict()}"
)

print(f"Raw respondents: {len(raw):,}")
print(f"Raw columns: {raw.shape[1]:,}")
print(f"Countries: {raw['cntry'].nunique(dropna=True)}")
print("Official ESS weight columns:", list(config.WEIGHT_COLUMNS))
display(weight_missing_raw.rename("missing_values").to_frame())

Raw respondents: 56,752
Raw columns: 674
Countries: 29
Official ESS weight columns: ['dweight', 'pspwght', 'pweight', 'anweight']


,missing_values
dweight,0
pspwght,0
pweight,0
anweight,0


## Step 3 — Construct identifiers, demographics, and official ESS weights

The output metadata follows the same general structure as the ESS8 curation.

- `ess_row_id` records the original row position in `ESS4e04_6.csv`, starting at 1.
- `ess_unique_id` combines country code and ESS respondent ID.
- Invalid ESS response codes outside the substantive range become missing.
- `education_3cat` follows Van Noord's Round 4 recoding of `edulvla`: 1–2 = lower, 3–4 = middle, and 5 = higher education.
- `urbanization = 6 - domicil`, so higher values indicate more urban surroundings.
- The four weight variables are copied directly from the raw ESS file and are not transformed.

The retained numeric demographic columns use the original ESS response coding. This keeps their interpretation transparent and consistent with the ESS8 curated output.

In [3]:
metadata = pd.DataFrame(index=raw.index)

metadata["ess_row_id"] = np.arange(1, len(raw) + 1)
metadata["idno"] = pd.to_numeric(raw["idno"], errors="coerce").astype("Int64")
metadata["cntry"] = raw["cntry"].astype("string")
metadata["country_name"] = metadata["cntry"].map(config.COUNTRY_LABELS)
metadata["ess_unique_id"] = (
    metadata["cntry"]
    + "_"
    + metadata["idno"].astype("string")
)

for column in ("agea", "gndr", "edulvla", "hinctnta", "rlgblg", "blgetmg"):
    minimum, maximum = config.METADATA_VALID_RANGES[column]
    metadata[column] = valid_range(raw[column], minimum, maximum)

metadata["education_3cat"] = np.nan
for category, source_values in config.EDUCATION_3CAT_MAP.items():
    metadata.loc[
        metadata["edulvla"].isin(source_values),
        "education_3cat",
    ] = category

minimum, maximum = config.METADATA_VALID_RANGES["domicil"]
domicil_clean = valid_range(raw["domicil"], minimum, maximum)
metadata["urbanization"] = config.URBANIZATION_REVERSE_CONSTANT - domicil_clean

for column in config.WEIGHT_COLUMNS:
    metadata[column] = pd.to_numeric(raw[column], errors="coerce")

metadata = metadata.loc[:, config.METADATA_COLUMNS_WITH_WEIGHTS]

assert metadata["country_name"].notna().all()
assert metadata["ess_unique_id"].notna().all()
assert metadata["ess_unique_id"].is_unique
assert metadata.loc[:, config.WEIGHT_COLUMNS].notna().all().all()

print("Metadata shape:", metadata.shape)
display(metadata.head())

Metadata shape: (56752, 17)


,ess_row_id,idno,cntry,country_name,ess_unique_id,agea,gndr,edulvla,education_3cat,hinctnta,rlgblg,urbanization,blgetmg,dweight,pspwght,pweight,anweight
0,1,10202,BE,Belgium,BE_10202,36.0,1.0,5.0,3.0,4.0,2.0,5.0,2.0,1.0074,0.823223,0.503773,0.414718
1,2,10203,BE,Belgium,BE_10203,26.0,2.0,5.0,3.0,7.0,2.0,5.0,2.0,1.0074,0.798609,0.503773,0.402318
2,3,10207,BE,Belgium,BE_10207,69.0,1.0,5.0,3.0,10.0,1.0,5.0,2.0,1.0074,0.778020,0.503773,0.391946
3,4,10208,BE,Belgium,BE_10208,77.0,2.0,5.0,3.0,7.0,1.0,5.0,2.0,1.0074,0.777735,0.503773,0.391802
4,5,10302,BE,Belgium,BE_10302,27.0,1.0,3.0,2.0,7.0,2.0,5.0,2.0,1.0074,0.960960,0.503773,0.484106


## Step 4 — Construct the 19 ESS Round 4 belief variables

The round-specific item definitions, valid response ranges, coding directions, and special recodes are stored in `src/ess4_config.py`.

Important Round 4 details include:

- `txearn` is recoded as 1 → 2, 2 → 1, and 3 → 3 before rescaling;
- response 4 is treated as missing for `earnpen` and `earnueb`;
- `dfincac` is not included in the final anti-egalitarianism scale, matching the supplied R script;
- all beliefs are placed on a 0–1 scale;
- multi-item beliefs use a strict row mean, so a missing constituent item makes that constructed belief missing.

Higher values follow the direction specified in the Round 4 source workflow and the common naming used in the curated datasets.

In [4]:
beliefs, coded_items = construct_belief_variables(
    raw,
    config.BELIEF_MAP,
    config.ITEM_CODING,
    item_value_recoding=config.ITEM_VALUE_RECODING,
    item_values_forced_missing=config.ITEM_VALUES_FORCED_MISSING,
)

assert beliefs.shape == (len(raw), config.EXPECTED_BELIEF_COUNT)
assert list(beliefs.columns) == list(config.BELIEF_COLUMNS)
assert list(coded_items.columns) == list(config.BELIEF_ITEM_COLUMNS)

belief_minimum = beliefs.min(skipna=True).min()
belief_maximum = beliefs.max(skipna=True).max()
assert belief_minimum >= 0.0
assert belief_maximum <= 1.0

print("Constructed belief variables:", beliefs.shape[1])
print("Cleaned constituent ESS items:", coded_items.shape[1])
print(f"Observed belief range: [{belief_minimum:.3f}, {belief_maximum:.3f}]")
display(beliefs.head())

Constructed belief variables: 19
Cleaned constituent ESS items: 39
Observed belief range: [0.000, 1.000]


,left_right_identification,gender_inequality,anti_lgbt,euroscepticism,anti_immigration,anti_egalitarianism,benefits_harm_economy,benefits_harm_society,welfare_chauvinism,anti_economic_interventionism,harsh_sentences,anti_militant_democracy,no_science_environment_solution,anti_government_spending,regressive_taxes,regressive_benefits,age_prejudice,authoritarianism,anti_libertarianism
0,0.7,0.50,0.75,0.8,1.000000,0.250,0.875,0.250,0.75,0.316667,0.25,0.75,0.25,0.5,0.5,0.0,0.6,0.48,0.40
1,0.6,0.75,0.25,0.4,0.444444,0.250,0.375,0.250,0.50,0.300000,0.75,0.25,0.50,0.5,0.5,0.0,0.3,0.60,0.44
2,0.8,0.75,0.00,0.2,0.333333,0.750,0.750,0.625,0.00,0.550000,0.50,0.75,0.75,0.5,0.0,0.0,0.6,0.56,0.20
3,0.6,0.75,0.50,0.4,0.555556,0.250,0.625,0.250,0.50,0.500000,0.50,0.25,0.25,0.5,0.0,0.0,0.2,0.68,0.44
4,0.5,0.75,0.00,0.0,0.666667,0.625,0.625,0.250,0.50,0.450000,0.25,0.75,0.25,0.5,0.5,0.5,0.0,0.64,0.36


## Step 5 — Combine metadata and beliefs, then apply the CCA sample rule

Before filtering on belief missingness, respondents are ordered by country and ESS respondent ID to mirror the ordering used in Van Noord's Round 4 cleaning script. The original CSV row number remains available in `ess_row_id`.

Respondents are retained when:

1. age is at least 18, or age is missing; and
2. no more than two of the 19 constructed beliefs are missing.

The two missingness columns are calculated from the constructed beliefs only.

In [5]:
curated_all = pd.concat([metadata, beliefs], axis=1)
curated_all = curated_all.sort_values(
    ["cntry", "idno"],
    kind="stable",
).reset_index(drop=True)

cca_initial = apply_cca_sample_rule(
    curated_all,
    config.BELIEF_COLUMNS,
    age_column="agea",
    minimum_age=config.MINIMUM_AGE,
    maximum_missing_beliefs=config.MAXIMUM_MISSING_BELIEFS,
)
cca_initial = cca_initial.reset_index(drop=True)

assert len(cca_initial) == config.EXPECTED_FINAL_N, (
    f"Expected {config.EXPECTED_FINAL_N:,} final rows, "
    f"found {len(cca_initial):,}."
)
assert cca_initial["cntry"].nunique() == config.EXPECTED_FINAL_COUNTRIES
assert cca_initial["n_belief_missing"].le(config.MAXIMUM_MISSING_BELIEFS).all()
assert (
    cca_initial["agea"].ge(config.MINIMUM_AGE)
    | cca_initial["agea"].isna()
).all()

print(f"Final CCA-ready respondents: {len(cca_initial):,}")
print(f"Countries retained: {cca_initial['cntry'].nunique()}")
print("Belief-missingness distribution:")
display(
    cca_initial["n_belief_missing"]
    .value_counts()
    .sort_index()
    .rename_axis("n_belief_missing")
    .rename("respondents")
    .to_frame()
)

Final CCA-ready respondents: 45,268
Countries retained: 29
Belief-missingness distribution:


,respondents
n_belief_missing,
0,30126
1,9561
2,5581


## Step 6 — Inspect ESS4 country counts and belief descriptives

These tables are diagnostic outputs for review. The final cross-round validation notebook will compare the respondent-level Python result with `reference/van_noord/ESS Round 4/data/df_ESS4.RData`.

In [6]:
country_counts = (
    cca_initial.groupby(["cntry", "country_name"], dropna=False)
    .size()
    .rename("N")
    .reset_index()
)

belief_summary = summarise_beliefs(
    cca_initial,
    config.BELIEF_COLUMNS,
)

print("Country-level final sample sizes:")
display(country_counts)

print("Python ESS4 belief-variable summary:")
display(belief_summary)

Country-level final sample sizes:


,cntry,country_name,N
0,BE,Belgium,1642
1,BG,Bulgaria,1352
2,CH,Switzerland,1597
3,CY,Cyprus,997
4,CZ,Czechia,1678
5,DE,Germany,2483
6,DK,Denmark,1461
7,EE,Estonia,1339
8,ES,Spain,2031
9,FI,Finland,1989


Python ESS4 belief-variable summary:


,belief_variable,N_reproduced,mean_reproduced,sd_reproduced
0,left_right_identification,41465,0.518816,0.225549
1,gender_inequality,45010,0.566380,0.257668
2,anti_lgbt,44516,0.335346,0.310386
3,euroscepticism,43342,0.466132,0.263316
4,anti_immigration,44297,0.481713,0.269713
5,anti_egalitarianism,44984,0.307890,0.212086
6,benefits_harm_economy,43533,0.515526,0.225427
7,benefits_harm_society,44693,0.419406,0.223687
8,welfare_chauvinism,44309,0.563802,0.252744
9,anti_economic_interventionism,44656,0.229808,0.158325


## Step 7 — Create the separate without-weights and with-weights datasets

The two files contain exactly the same respondents, identifiers, demographics, belief values, and missingness counts.

The only difference is that the with-weights version includes the four official ESS columns immediately after `blgetmg`:

- `dweight`
- `pspwght`
- `pweight`
- `anweight`

No observations are duplicated and no belief values are multiplied by weights.

In [7]:
cca_without_weights, cca_with_weights = split_weighted_and_unweighted(
    cca_initial,
    config.WEIGHT_COLUMNS,
    insert_after=config.WEIGHT_INSERT_AFTER,
)

cca_without_weights = cca_without_weights.loc[
    :, config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS
].copy()
cca_with_weights = cca_with_weights.loc[
    :, config.OUTPUT_COLUMNS_WITH_WEIGHTS
].copy()

assert cca_without_weights.shape == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
assert cca_with_weights.shape == config.EXPECTED_WITH_WEIGHTS_SHAPE

validate_weight_split(
    cca_without_weights,
    cca_with_weights,
    config.WEIGHT_COLUMNS,
    require_complete_weights=True,
)

print("Without-weights shape:", cca_without_weights.shape)
print("With-weights shape:", cca_with_weights.shape)
print("The datasets differ only by the four official weight columns.")

Without-weights shape: (45268, 34)
With-weights shape: (45268, 38)
The datasets differ only by the four official weight columns.


## Step 8 — Save the two ESS4 result files

The outputs are written to `data/processed/` using the agreed filenames. Existing files with the same names are replaced only when this cell is deliberately run.

In [8]:
cca_without_weights.to_csv(WITHOUT_WEIGHTS_PATH, index=False)
cca_with_weights.to_csv(WITH_WEIGHTS_PATH, index=False)

print("Saved:")
print(" -", WITHOUT_WEIGHTS_PATH)
print(" -", WITH_WEIGHTS_PATH)

Saved:
 - /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess4_cca_initial_beliefs_without_weights.csv
 - /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess4_cca_initial_beliefs_with_weights.csv


## Step 9 — Read the saved files back and perform final integrity checks

Reading the CSV files back from disk catches problems caused by column order, serialization, or accidental index writing. The final checks confirm dimensions, respondent identity, belief equality, missingness equality, and complete official weights.

In [9]:
saved_without = pd.read_csv(WITHOUT_WEIGHTS_PATH, low_memory=False)
saved_with = pd.read_csv(WITH_WEIGHTS_PATH, low_memory=False)

assert saved_without.shape == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
assert saved_with.shape == config.EXPECTED_WITH_WEIGHTS_SHAPE
assert list(saved_without.columns) == list(config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS)
assert list(saved_with.columns) == list(config.OUTPUT_COLUMNS_WITH_WEIGHTS)

validate_weight_split(
    saved_without,
    saved_with,
    config.WEIGHT_COLUMNS,
    require_complete_weights=True,
)

assert saved_without["ess_unique_id"].is_unique
assert saved_with["ess_unique_id"].is_unique
assert saved_without["ess_unique_id"].equals(saved_with["ess_unique_id"])

print("ESS Round 4 curation completed successfully.")
print(f"Raw respondents: {config.EXPECTED_RAW_N:,}")
print(f"Final respondents: {len(saved_without):,}")
print(f"Countries: {saved_without['cntry'].nunique()}")
print(f"Belief variables: {len(config.BELIEF_COLUMNS)}")
print(f"Without-weights columns: {saved_without.shape[1]}")
print(f"With-weights columns: {saved_with.shape[1]}")
print("Missing official weights:")
print(saved_with.loc[:, config.WEIGHT_COLUMNS].isna().sum().to_dict())

ESS Round 4 curation completed successfully.
Raw respondents: 56,752
Final respondents: 45,268
Countries: 29
Belief variables: 19
Without-weights columns: 34
With-weights columns: 38
Missing official weights:
{'dweight': 0, 'pspwght': 0, 'pweight': 0, 'anweight': 0}
